# Phase 3: Stratified Train-Test Splitting & Leakage Prevention

This notebook performs stratified train-test splitting on the cleaned California (CA) 2018 ACSIncome dataset.

### Protocol Requirements:
1. Load `X_clean.csv` and `y_clean.csv`.
2. Perform stratified split: `test_size=0.20`, `random_state=42`, `stratify=y`.
3. Save partitioned datasets to `data/processed/`:
   - `X_train.csv`
   - `X_test.csv`
   - `y_train.csv`
   - `y_test.csv`
4. Inspect row counts and target proportions across train and test sets.
5. Save verification metrics to `results/03_train_test_split_summary.csv`.
6. Enforce strict data leakage isolation: no preprocessing or modeling fitted on the full dataset.


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Resolve paths
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name in ['notebooks', 'src'] else NOTEBOOK_DIR

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

X_CLEAN_PATH = DATA_PROCESSED_DIR / "X_clean.csv"
Y_CLEAN_PATH = DATA_PROCESSED_DIR / "y_clean.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Clean features file: {X_CLEAN_PATH} (Exists: {X_CLEAN_PATH.exists()})")
print(f"Clean target file:   {Y_CLEAN_PATH} (Exists: {Y_CLEAN_PATH.exists()})")


Project root: D:\Github Repo\C-Square
Clean features file: D:\Github Repo\C-Square\data\processed\X_clean.csv (Exists: True)
Clean target file:   D:\Github Repo\C-Square\data\processed\y_clean.csv (Exists: True)


In [2]:
# Step 1: Load X_clean.csv and y_clean.csv
X = pd.read_csv(X_CLEAN_PATH)
y = pd.read_csv(Y_CLEAN_PATH)

if isinstance(y, pd.DataFrame):
    y_series = y.squeeze()
else:
    y_series = y

print("=" * 70)
print("DATASET LOADED:")
print("=" * 70)
print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape:  {y_series.shape}")
print(f"Target column name:     {y_series.name}")


DATASET LOADED:
Feature matrix X shape: (192755, 10)
Target vector y shape:  (192755,)
Target column name:     income_above_50k


In [3]:
# Step 2: Split data into training and testing sets
# test_size = 0.20, random_state = 42, stratify = y
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_series,
    test_size=0.20,
    random_state=42,
    stratify=y_series
)

print("=" * 70)
print("TRAIN-TEST PARTITIONING COMPLETE:")
print("=" * 70)
print(f"Training features (X_train): {X_train.shape[0]:,} rows x {X_train.shape[1]} columns")
print(f"Testing features  (X_test):  {X_test.shape[0]:,} rows x {X_test.shape[1]} columns")
print(f"Training target   (y_train): {y_train.shape[0]:,} rows")
print(f"Testing target    (y_test):  {y_test.shape[0]:,} rows")


TRAIN-TEST PARTITIONING COMPLETE:
Training features (X_train): 154,204 rows x 10 columns
Testing features  (X_test):  38,551 rows x 10 columns
Training target   (y_train): 154,204 rows
Testing target    (y_test):  38,551 rows


In [4]:
# Step 3: Save partitioned datasets
X_train_path = DATA_PROCESSED_DIR / "X_train.csv"
X_test_path = DATA_PROCESSED_DIR / "X_test.csv"
y_train_path = DATA_PROCESSED_DIR / "y_train.csv"
y_test_path = DATA_PROCESSED_DIR / "y_test.csv"

print(f"Saving X_train to: {X_train_path}")
X_train.to_csv(X_train_path, index=False)

print(f"Saving X_test to:  {X_test_path}")
X_test.to_csv(X_test_path, index=False)

print(f"Saving y_train to: {y_train_path}")
y_train.to_frame(name="income_above_50k").to_csv(y_train_path, index=False)

print(f"Saving y_test to:  {y_test_path}")
y_test.to_frame(name="income_above_50k").to_csv(y_test_path, index=False)

print("Partition files saved successfully!")


Saving X_train to: D:\Github Repo\C-Square\data\processed\X_train.csv


Saving X_test to:  D:\Github Repo\C-Square\data\processed\X_test.csv
Saving y_train to: D:\Github Repo\C-Square\data\processed\y_train.csv
Saving y_test to:  D:\Github Repo\C-Square\data\processed\y_test.csv
Partition files saved successfully!


In [5]:
# Step 4: Print row counts and target distributions in training and testing sets
train_counts = y_train.value_counts().sort_index()
train_pcts = (train_counts / len(y_train)) * 100

test_counts = y_test.value_counts().sort_index()
test_pcts = (test_counts / len(y_test)) * 100

print("=" * 70)
print("TARGET DISTRIBUTION IN TRAINING SET:")
print("=" * 70)
print(f"Class 0 (<= $50k): {train_counts[0]:,} ({train_pcts[0]:.4f}%)")
print(f"Class 1 (> $50k):  {train_counts[1]:,} ({train_pcts[1]:.4f}%)")
print(f"Total rows:        {len(y_train):,}")

print("\n" + "=" * 70)
print("TARGET DISTRIBUTION IN TESTING SET:")
print("=" * 70)
print(f"Class 0 (<= $50k): {test_counts[0]:,} ({test_pcts[0]:.4f}%)")
print(f"Class 1 (> $50k):  {test_counts[1]:,} ({test_pcts[1]:.4f}%)")
print(f"Total rows:        {len(y_test):,}")

# Step 5: Verify that the target proportions are identical within tolerance
pct_diff_class0 = abs(train_pcts[0] - test_pcts[0])
pct_diff_class1 = abs(train_pcts[1] - test_pcts[1])

print("\n" + "=" * 70)
print("STRATIFICATION VERIFICATION:")
print("=" * 70)
print(f"Class 0 proportion difference: {pct_diff_class0:.6f}%")
print(f"Class 1 proportion difference: {pct_diff_class1:.6f}%")

assert pct_diff_class0 < 0.01, "Stratification failure on class 0!"
assert pct_diff_class1 < 0.01, "Stratification failure on class 1!"
print("Verification check passed: Stratification perfectly preserved target proportions!")


TARGET DISTRIBUTION IN TRAINING SET:
Class 0 (<= $50k): 90,088 (58.4213%)
Class 1 (> $50k):  64,116 (41.5787%)
Total rows:        154,204

TARGET DISTRIBUTION IN TESTING SET:
Class 0 (<= $50k): 22,522 (58.4213%)
Class 1 (> $50k):  16,029 (41.5787%)
Total rows:        38,551

STRATIFICATION VERIFICATION:
Class 0 proportion difference: 0.000000%
Class 1 proportion difference: 0.000000%
Verification check passed: Stratification perfectly preserved target proportions!


In [6]:
# Step 6: Save a split summary to results/03_train_test_split_summary.csv
full_counts = y_series.value_counts().sort_index()
full_pcts = (full_counts / len(y_series)) * 100

summary_records = [
    {
        "Dataset name": "Full Dataset",
        "Number of rows": int(len(y_series)),
        "Number of target class 0": int(full_counts[0]),
        "Percentage of target class 0": round(float(full_pcts[0]), 4),
        "Number of target class 1": int(full_counts[1]),
        "Percentage of target class 1": round(float(full_pcts[1]), 4),
    },
    {
        "Dataset name": "Training Set",
        "Number of rows": int(len(y_train)),
        "Number of target class 0": int(train_counts[0]),
        "Percentage of target class 0": round(float(train_pcts[0]), 4),
        "Number of target class 1": int(train_counts[1]),
        "Percentage of target class 1": round(float(train_pcts[1]), 4),
    },
    {
        "Dataset name": "Testing Set",
        "Number of rows": int(len(y_test)),
        "Number of target class 0": int(test_counts[0]),
        "Percentage of target class 0": round(float(test_pcts[0]), 4),
        "Number of target class 1": int(test_counts[1]),
        "Percentage of target class 1": round(float(test_pcts[1]), 4),
    },
]

summary_df = pd.DataFrame(summary_records)
print(summary_df.to_string(index=False))

summary_csv_path = RESULTS_DIR / "03_train_test_split_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)
print(f"\nSplit summary successfully saved to: {summary_csv_path}")


Dataset name  Number of rows  Number of target class 0  Percentage of target class 0  Number of target class 1  Percentage of target class 1
Full Dataset          192755                    112610                       58.4213                     80145                       41.5787
Training Set          154204                     90088                       58.4213                     64116                       41.5787
 Testing Set           38551                     22522                       58.4213                     16029                       41.5787

Split summary successfully saved to: D:\Github Repo\C-Square\results\03_train_test_split_summary.csv


In [7]:
# Step 10: Confirm that all output files exist
required_files = [
    DATA_PROCESSED_DIR / "X_train.csv",
    DATA_PROCESSED_DIR / "X_test.csv",
    DATA_PROCESSED_DIR / "y_train.csv",
    DATA_PROCESSED_DIR / "y_test.csv",
    RESULTS_DIR / "03_train_test_split_summary.csv",
]

print("=" * 70)
print("CONFIRMING PHASE 3 DELIVERABLES:")
print("=" * 70)
all_found = True
for f in required_files:
    exists = f.exists()
    size_str = f"({f.stat().st_size:,} bytes)" if exists else ""
    print(f" - {f.name:36s}: {'[EXISTS]' if exists else '[MISSING]'} {size_str}")
    if not exists:
        all_found = False

assert all_found, "One or more required files are missing!"
print("\nAll Phase 3 deliverables verified successfully!")


CONFIRMING PHASE 3 DELIVERABLES:
 - X_train.csv                         : [EXISTS] (7,352,026 bytes)
 - X_test.csv                          : [EXISTS] (1,837,995 bytes)
 - y_train.csv                         : [EXISTS] (462,630 bytes)
 - y_test.csv                          : [EXISTS] (115,671 bytes)
 - 03_train_test_split_summary.csv     : [EXISTS] (283 bytes)

All Phase 3 deliverables verified successfully!


## Methodological Notice: Data Isolation & Leakage Prevention

> ### **CRITICAL ML PRINCIPLE: STRICT LEAKAGE PREVENTION**
> All future preprocessing steps—including:
> - Category frequency thresholding,
> - Rare category aggregation (`OCCP`, `POBP`),
> - One-hot encoding / target encoding,
> - Numeric scaling / binning,
> - Contingency table construction and $\chi^2$ hypothesis tests, and
> - Machine learning estimator training
>
> **MUST be fitted exclusively on `X_train`** (and `y_train`).
>
> `X_test` must remain completely untouched during exploratory modeling, feature selection, and transformation fitting. The fitted transformers learned from `X_train` will only be applied to `X_test` to transform it at final evaluation time, ensuring authentic generalization performance and unbiased statistical test validation.
